# Getting Started with StarLayer

StarLayer is a Python wrapper built on rdflib and pyshacl (and hermit, etc. for reasoning, pending) to support the 1.2 draft standards for RDF, SPARQL and SHACL.  This guide is deliberately simple, to demonstrate the broad capabilities of StarLayer and to show the base transition from rdflib/pyshacl to StarLayer.   High level introductions provided below provide links to aspects of the guide that go deeper into specific topics.

These user guides assume a working knolwedge of RDF 1.1, SPARQL 1.1 and SHACL 1.1 in the python environment using rdflib and pyshacl.  


## 1. Install

This is the supported public install path. It brings in the graph, SPARQL, and SHACL components together. If you're working from a local checkout for development instead, see the root [README](../../README.md)'s "Development" section.

Because StarLayer is under development and evolving in-line with the draft standards, it is helpful to periodically refresh the install from the github repository.  

In [1]:
try:
    import starlayergraph  # noqa: F401
    print("starlayer is already installed - skipping install.")
except ImportError:
    %pip install "git+https://github.com/hidden-graph/starlayer.git"

### Getting a fresh version

# The check above skips installing if `starlayergraph` is already importable.
# It won't relfect subsequent changes. To force a clean reinstall from the repo, uncomment and run:

# %pip uninstall -y starlayer
# %pip install --no-cache-dir "git+https://github.com/hidden-graph/starlayer.git"



starlayer is already installed - skipping install.


In [2]:
from starlayer import StarLayerGraph, Namespace, StarShaclValidator

EX = Namespace("http://example.org/")

## 2. Using StarLayerGraph

`StarLayerGraph` is a drop-in replacement for `rdflib.Graph`.  StarLayerGraph functions such as `parse()`/`serialize()` operate the same as rdflib when operating on RDF 1.1 graphs.   

The [Graphs guide](02-graphs.ipynb) covers RDF 1.2-specific content (triple terms, reification, direction-tagged literals.).

The examples in the user guide are presented in the Turtle serialization format.  The the [serialization formats guide](02b-graphs-serialization-formats.ipynb) covers additional supported serialization formats.

In [3]:
# StarLayerGraph is a drop-in replacement for rdflib.Graph() - construct it the same way.
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ;
      ex:name "Alice" ;
      ex:knows ex:bob .
""", format="turtle")

print("triples:", len(g))
print(g.serialize(format="turtle"))

triples: 3
@prefix ex: <http://example.org/> .

ex:alice a ex:Person ;
    ex:knows ex:bob ;
    ex:name "Alice" .




## 3. Querying a StarLayerGraph

The `.query()` function runs a SPARQL query against the graph. This is the same functionality as rdflib. See the [SPARQL guide](03-sparql.ipynb) for RDF-1.2-aware query functions.  

In [4]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ;
      ex:name "Alice" ;
      ex:knows ex:bob .
""", format="turtle")

rows = g.query("""
    PREFIX ex: <http://example.org/>
    SELECT ?name WHERE { ?person ex:name ?name }
""")
for row in rows:
    print(row.name)

Alice


## 4. Validation with StarShacl

A SHACL shape graph can be used to describe the constraints within a graph. `StarShaclValidator().validate()` checks for conformance of a data graph against a shapes graph.  The data graph, the shapes graph and the results are all StarLayerGraphs. 

See the [SHACL shapes guide](04-shacl-shapes.ipynb) for an in-depth overview of StarShacl capabilities.   

In [5]:
data = StarLayerGraph()
data.bind("ex", EX)
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ;
      ex:name "Alice" ;
      ex:knows ex:bob .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes)
print("conforms:", result.conforms)

# result.report_graph is itself a StarLayerGraph - pretty-print it as turtle12
print(result.report_graph.serialize(format="turtle12"))

conforms: True
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

_:N75f51186ac9f4315961c42c0f490f905 a sh:ValidationReport ;
    sh:conformanceDisallows sh:Violation, sh:Info, sh:Warning ;
    sh:conforms true .



## Where to go next


1. **Getting Started** — this guide.
2. **[Graphs](02-graphs.ipynb)** — Enhancements to rdflib to handle RDF 1.2 changes including `TripleTerm`/`DirLangString` semantics.
   - 2.a **[Working with datasets](02a-graphs-datasets.ipynb)** — `StarLayerDataset`, multiple named graphs in one store.
3. **[SPARQL](03-sparql.ipynb)** — Enhancements to reflect SPARQL 1.2 changes.
   - 3.a **[SPARQL rules (pending)](03a-sparql-rules-pending.md)** — SPARQL-RL (SRL), a separate Datalog-style rules language.
4. **[SHACL shapes](04-shacl-shapes.ipynb)** — Enhancements to pyshacl to reflect SHACL 1.2 changes.  Includes overview of four processing modes (`validate()`, `apply_rules()`, `evaluate()`, `extract_subgraph()`).
   - 4.a **[SHACL node expressions](04a-shacl-node-expressions.ipynb)** — the SHACL 1.2 node-expression vocabulary including custom node expression functions.
   - 4.b **[SHACL inference rules](04b-shacl-inference-rules.ipynb)** — SHACL rules including execution ordering, rule sets, provenance.
   - 4.c **[SHACL and SPARQL](04c-shacl-and-sparql.ipynb)** — Using SHACL and SPARQL together. 
   - 4.d **[SHACL UI](04d-shacl-ui.ipynb)** — Using SHACL to hold presentation metadata.
   - 4.e **[SHACL profiling](04e-shacl-profiling.ipynb)** — Packaging SHACL shape graphs.
   - 4.f **[SHACL subgraph extraction](04f-shacl-subgraph-extraction.ipynb)** — extract data that conforms to a SHACL shape.
5. **Other**
   - 5.a **[Serialization formats](02b-graphs-serialization-formats.ipynb)** — all supported RDF 1.2 formats.
   - 5.b **[Working with backend graph databases](06-backend-graph-databases.ipynb)** — Oxigraph, Fuseki, and SQLAlchemy-backed storage.
   - 5.c **[Inferencing](05-inferencing.ipynb)** — RDFS subclass reasoning and OWL reasoning (pending).
   - 5.d **[SPARQL queries as RDF](03b-sparql-query-as-rdf.ipynb)** — encoding, editing, and validating a query itself as an RDF graph.
   - 5.e **[Canonical hashing and graph comparison](02c-graphs-canonical-hashing.ipynb)** — RDFC-1.0 canonicalization/hashing and graph isomorphism.